In [15]:
# Install required packages (run once in Colab)
!pip install dash pyngrok -q

In [8]:
# Import required libraries
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the SpaceX data
URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
)
spacex_df = pd.read_csv(URL)

max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Launch sites
launch_sites = spacex_df['Launch Site'].unique()

# Create Dash app
app = dash.Dash(__name__)

# App layout
app.layout = html.Div(children=[

    html.H1(
        'SpaceX Launch Records Dashboard',
        style=
            {
                'textAlign': 'center',
                'color': '#503D36',
                'font-size': 40
            }
    ),

    # TASK 1: Dropdown
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'}
        ] + [
            {'label': site, 'value': site}
            for site in launch_sites
        ],
        value='ALL',
        placeholder='Select a Launch Site',
        searchable=True
    ),

    html.Br(),

    # TASK 2: Pie chart
    html.Div(
        dcc.Graph(id='success-pie-chart')
    ),

    html.Br(),

    html.P("Payload range (Kg):"),

    # TASK 3: Range Slider
    dcc.RangeSlider(
        id='payload-slider',
        min=min_payload,
        max=max_payload,
        step=1000,
        value=[min_payload, max_payload],
        marks=
            {
                int(min_payload): str(int(min_payload)),
                int(max_payload): str(int(max_payload))
            }
    ),

    html.Br(),

    # TASK 4: Scatter chart
    html.Div(
        dcc.Graph(id='success-payload-scatter-chart')
    )

])


# TASK 2 Callback
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def get_pie_chart(selected_site):

    if selected_site == 'ALL':

        fig = px.pie(
            spacex_df,
            names='Launch Site',
            values='class',
            title='Total Successful Launches by Site'
        )

    else:

        filtered_df = spacex_df[
            spacex_df['Launch Site'] == selected_site
        ]

        fig = px.pie(
            filtered_df,
            names='class',
            title=f'Success vs Failure for {selected_site}'
        )

    return fig


# TASK 4 Callback
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [
        Input('site-dropdown', 'value'),
        Input('payload-slider', 'value')
    ]
)
def update_scatter(selected_site, payload_range):

    low, high = payload_range

    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) &
        (spacex_df['Payload Mass (kg)'] <= high)
    ]

    if selected_site != 'ALL':
        filtered_df = filtered_df[
            filtered_df['Launch Site'] == selected_site
        ]

    fig = px.scatter(
        filtered_df,
        x='Payload Mass (kg)',
        y='class',
        color='Booster Version Category',
        title='Payload vs Launch Outcome'
    )

    return fig

In [9]:
print('Dash app object initialized successfully.')

Dash app object initialized successfully.


In [25]:
from pyngrok import ngrok
import threading

ngrok.kill() # Terminate any existing tunnels from this process
ngrok.set_auth_token(authtoken) # Authenticate ngrok

public_url = ngrok.connect(8050)
print("Dash URL:", public_url)

thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=8050, debug=False)
)
thread.start()

Dash URL: NgrokTunnel: "https://paddle-unvaried-ship.ngrok-free.dev" -> "http://localhost:8050"
Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


In [11]:
print(app.layout)

Div([H1(children='SpaceX Launch Records Dashboard', style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}), Dropdown(options=[{'label': 'All Sites', 'value': 'ALL'}, {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'}, {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'}, {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'}, {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}], value='ALL', searchable=True, placeholder='Select a Launch Site', id='site-dropdown'), Br(None), Div(Graph(id='success-pie-chart')), Br(None), P('Payload range (Kg):'), RangeSlider(min=0.0, max=9600.0, step=1000, marks={0: '0', 9600: '9600'}, value=[0.0, 9600.0], id='payload-slider'), Br(None), Div(Graph(id='success-payload-scatter-chart'))])


In [12]:
print(spacex_df.head())

   Unnamed: 0  Flight Number  Launch Site  class  Payload Mass (kg)  \
0           0              1  CCAFS LC-40      0                0.0   
1           1              2  CCAFS LC-40      0                0.0   
2           2              3  CCAFS LC-40      0              525.0   
3           3              4  CCAFS LC-40      0              500.0   
4           4              5  CCAFS LC-40      0              677.0   

  Booster Version Booster Version Category  
0  F9 v1.0  B0003                     v1.0  
1  F9 v1.0  B0004                     v1.0  
2  F9 v1.0  B0005                     v1.0  
3  F9 v1.0  B0006                     v1.0  
4  F9 v1.0  B0007                     v1.0  


In [18]:
spacex_df.columns

Index(['Unnamed: 0', 'Flight Number', 'Launch Site', 'class',
       'Payload Mass (kg)', 'Booster Version', 'Booster Version Category'],
      dtype='object')

In [28]:
for child in app.layout.children:
    print(type(child))
    try:
        print(child.children)
    except:
        pass

<class 'dash.html.H1.H1'>
SpaceX Launch Records Dashboard
<class 'dash.dcc.Dropdown.Dropdown'>
<class 'dash.html.Br.Br'>
None
<class 'dash.html.Div.Div'>
Graph(id='success-pie-chart')
<class 'dash.html.Br.Br'>
None
<class 'dash.html.P.P'>
Payload range (Kg):
<class 'dash.dcc.RangeSlider.RangeSlider'>
<class 'dash.html.Br.Br'>
None
<class 'dash.html.Div.Div'>
Graph(id='success-payload-scatter-chart')


In [30]:
print(app.layout.children[0].children)

SpaceX Launch Records Dashboard


In [13]:
'app' in globals()

True

In [18]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8055)

print(public_url)

NgrokTunnel: "https://paddle-unvaried-ship.ngrok-free.dev" -> "http://localhost:8055"


In [17]:
import requests

r = requests.get("http://127.0.0.1:8050")

print(r.status_code)

INFO:werkzeug:127.0.0.1 - - [23/Jun/2026 05:24:40] "GET / HTTP/1.1" 200 -


200
